# WHBot — Google Colab

This notebook keeps **runtime/dependencies separate from GitHub code updates**.

- Store credentials in **Colab Secrets**.
- Install system/Python dependencies once per runtime.
- Clone or update the `gofile` branch without deleting the runtime data.
- `/content/data`, `/content/downloads`, and `.pyrogram` are preserved when code is updated.

### Required Colab Secrets
`API_ID`, `API_HASH`, `BOT_TOKEN`, `DOWNLOAD_CHANNEL_ID`

Optional: `CHANNEL_ID`


## 1. Runtime dependencies
Run this once after starting a new Colab runtime. You do **not** need to run it every time you pull new GitHub code.


In [ ]:
!command -v ffprobe >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq ffmpeg)
print('✓ System dependency check complete.')


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/whbot')
REPO_URL = 'https://github.com/Telebotfaroff/whbot.git'
BRANCH = 'gofile'

if (REPO_DIR / '.git').exists():
    print('Updating existing checkout...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    print('Cloning repository...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)

(REPO_DIR / '.pyrogram').mkdir(parents=True, exist_ok=True)
Path('/content/data').mkdir(parents=True, exist_ok=True)
Path('/content/downloads').mkdir(parents=True, exist_ok=True)

actual_branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
print(f'✓ Code ready: {actual_branch}')
print('✓ Existing runtime data/downloads/session were preserved.')


## 2. Clone / update GitHub code
Run this whenever you want the latest code. It **does not delete the Colab runtime** and does not remove `/content/data` or `/content/downloads`.



In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/whbot')
REPO_URL = 'https://github.com/Telebotfaroff/whbot.git'
BRANCH = 'gofile'

if (REPO_DIR / '.git').exists():
    print('Updating existing checkout...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    print('Cloning repository...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)

(REPO_DIR / '.pyrogram').mkdir(parents=True, exist_ok=True)
Path('/content/data').mkdir(parents=True, exist_ok=True)
Path('/content/downloads').mkdir(parents=True, exist_ok=True)

actual_branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
print(f'✓ Code ready: {actual_branch}')
print('✓ Existing runtime data/downloads/session were preserved.')


## 3. Install/update Python dependencies
Run this after the GitHub update **only if `requirements.txt` changed** (or if this is a new runtime).



In [ ]:
%cd /content/whbot
!pip install -q -r requirements.txt
print('✓ Python dependencies are ready.')


## 4. Load credentials from Colab Secrets
In Colab, open the **Secrets** panel and add the required values. The notebook does not store the credentials in the `.ipynb` file.



In [ ]:
from pathlib import Path
from google.colab import userdata

def secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception as exc:
        raise RuntimeError(f'Could not read Colab Secret {name!r}: {exc}') from exc
    value = (value or '').strip()
    if required and not value:
        raise ValueError(f'Missing required Colab Secret: {name}')
    return value

API_ID = secret('API_ID')
API_HASH = secret('API_HASH')
BOT_TOKEN = secret('BOT_TOKEN')
DOWNLOAD_CHANNEL_ID = secret('DOWNLOAD_CHANNEL_ID')
CHANNEL_ID = secret('CHANNEL_ID', required=False)

if not API_ID.isdigit():
    raise ValueError('API_ID must contain digits only.')
if ':' not in BOT_TOKEN:
    raise ValueError('BOT_TOKEN has an invalid format.')

print('✓ Colab Secrets loaded successfully.')
print(f'✓ API_ID: {API_ID}')
print(f'✓ DOWNLOAD_CHANNEL_ID: {DOWNLOAD_CHANNEL_ID}')
print(f'✓ CHANNEL_ID: {CHANNEL_ID or "not set (optional)"}')


## 5. Generate runtime `.env`
This writes secrets only to the current Colab runtime at `/content/whbot/.env`. The file is not committed back to GitHub.



In [ ]:
from pathlib import Path

env = '\n'.join([
    'API_ID=' + API_ID,
    'API_HASH=' + API_HASH,
    'BOT_TOKEN=' + BOT_TOKEN,
    'DOWNLOAD_CHANNEL_ID=' + DOWNLOAD_CHANNEL_ID,
    'CHANNEL_ID=' + CHANNEL_ID,
    'WATCHHENTAI_BASE_URL=https://watchhentai.net',
    'GOFILE_UPLOAD_URL=https://upload.gofile.io/uploadfile',
    'MAX_MERGE_VIDEOS=100',
    'MAX_MERGE_FILE_GB=20',
    'MERGE_PRESET=veryfast',
    'MERGE_CRF=23',
    'MERGE_AUDIO_BITRATE=128k',
    'DOWNLOAD_DIR=/content/downloads',
    'WH_DB_PATH=/content/data/watchhentai.db',
    'SCHEDULER_ENABLED=false',
    'CRAWL_INTERVAL=1800',
    'CRAWL_PAGES=1',
    'PUBLISH_LIMIT=20',
    'INITIAL_CRAWL_PAGES=0',
]) + '\n'

Path('/content/whbot/.env').write_text(env)
Path('/content/whbot/.pyrogram').mkdir(parents=True, exist_ok=True)
print('✓ Runtime configuration generated.')


## 6. Configuration summary

- Code branch: `gofile`
- Database: `/content/data/watchhentai.db` — preserved during code updates
- Downloads: `/content/downloads` — preserved during code updates
- Pyrogram session: `/content/whbot/.pyrogram` — preserved during code updates
- Automatic scheduler: disabled
- `/crawl`: crawl series pages into SQLite
- `/download`: preview and download one complete series; delivery is through GoFile
- `/auto`: process the series queue sequentially; delivery is through GoFile
- `/merge`: collect videos, re-encode them with FFmpeg, and upload the merged result to GoFile
- Series downloads retain successful episode files until the merge session is completed or cancelled



## 7. Start the bot



In [ ]:
%cd /content/whbot
import os
os.environ['PYTHONUNBUFFERED'] = '1'
print('===== WHBot live logs enabled =====')
print('Resolver logs: [resolver] | Downloader: [downloader] | Upload: [upload]')
!python -u main.py


## 8. Update code without restarting the runtime
Stop the running bot first, then run the **Clone / update GitHub code** cell again. If `requirements.txt` changed, run the dependency cell too, then run the `.env` generation cell and start the bot again.

